# 💻 Case Study: RAG-Powered Code Assistant

## What We're Building

A code assistant that answers questions about **your own codebase**.

```
You:    "How do I call the auth function?"
Bot:    "In auth.py line 42: call authenticate(user, token)"
```

Real use cases: onboarding new engineers, internal dev tools, code review helpers.

## What's Different for Code?

- Chunk by **function/class**, not by sentence
- Include **filename + line number** in metadata
- Questions are about *how to use* code, not facts

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


In [2]:
# Simulated code snippets from your codebase
# Each entry: the code text + where it lives

code_chunks = [
    {
        "file": "auth.py", "line": 12,
        "code": "def authenticate(username, token):\n    \"\"\"Check if token is valid for user. Returns True/False.\"\"\"\n    return token_store.get(username) == token"
    },
    {
        "file": "auth.py", "line": 25,
        "code": "def create_token(username):\n    \"\"\"Generate a new auth token for user. Call after login.\"\"\"\n    token = secrets.token_hex(16)\n    token_store[username] = token\n    return token"
    },
    {
        "file": "database.py", "line": 8,
        "code": "def get_user(user_id):\n    \"\"\"Fetch user record by ID. Returns dict or None.\"\"\"\n    return db.query('SELECT * FROM users WHERE id = ?', [user_id])"
    },
    {
        "file": "database.py", "line": 20,
        "code": "def save_user(user_data):\n    \"\"\"Insert or update a user record.\"\"\"\n    db.execute('INSERT OR REPLACE INTO users VALUES (?)', [user_data])"
    },
    {
        "file": "api.py", "line": 45,
        "code": "@app.route('/login', methods=['POST'])\ndef login():\n    \"\"\"Login endpoint. Expects JSON: {username, password}. Returns token.\"\"\"\n    user = get_user(request.json['username'])\n    token = create_token(user['id'])\n    return {'token': token}"
    },
]

# Embed code as text — the model understands code too!
texts = [c["code"] for c in code_chunks]
embeddings = model.encode(texts)

print(f"Indexed {len(code_chunks)} code snippets")

Indexed 5 code snippets


In [3]:
# Code-aware search

def search_code(question, top_k=2):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"Question: '{question}'")
    print("-" * 55)
    for rank, idx in enumerate(top_idx, 1):
        c = code_chunks[idx]
        print(f"  {rank}. [{scores[idx]:.3f}] {c['file']} line {c['line']}")
        # Show just the first line (signature)
        first_line = c['code'].split('\n')[0]
        print(f"     {first_line}")
    print()

search_code("how do I log in a user?")
search_code("check if token is valid")
search_code("look up a user from the database")

Question: 'how do I log in a user?'
-------------------------------------------------------
  1. [0.421] auth.py line 12
     def authenticate(username, token):
  2. [0.414] api.py line 45
     @app.route('/login', methods=['POST'])

Question: 'check if token is valid'
-------------------------------------------------------
  1. [0.583] auth.py line 12
     def authenticate(username, token):
  2. [0.315] auth.py line 25
     def create_token(username):

Question: 'look up a user from the database'
-------------------------------------------------------
  1. [0.554] database.py line 8
     def get_user(user_id):
  2. [0.403] database.py line 20
     def save_user(user_data):



In [4]:
# Full prompt for code Q&A

def build_code_prompt(question, top_k=2):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    context_parts = []
    for idx in top_idx:
        c = code_chunks[idx]
        context_parts.append(f"# {c['file']} (line {c['line']})\n{c['code']}")

    context = "\n\n".join(context_parts)

    prompt = f"""You are a code assistant. Use only the code below to answer the question.

{context}

Question: {question}
Answer:"""
    return prompt


question = "How do I create an auth token for a user?"
print(build_code_prompt(question))

You are a code assistant. Use only the code below to answer the question.

# auth.py (line 25)
def create_token(username):
    """Generate a new auth token for user. Call after login."""
    token = secrets.token_hex(16)
    token_store[username] = token
    return token

# auth.py (line 12)
def authenticate(username, token):
    """Check if token is valid for user. Returns True/False."""
    return token_store.get(username) == token

Question: How do I create an auth token for a user?
Answer:


## How to Index a Real Codebase

```python
import ast, os

chunks = []

for py_file in Path("my_project").rglob("*.py"):
    source = py_file.read_text()
    tree   = ast.parse(source)

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            # Extract the function/class source
            start = node.lineno
            end   = node.end_lineno
            code  = '\n'.join(source.split('\n')[start-1:end])

            chunks.append({
                "file": str(py_file),
                "line": start,
                "code": code
            })
```

Same pattern works for JavaScript, TypeScript, Java — just swap the parser.